# ClimaCity Paris -- 02. Pipeline DataFrame (Jour 1 après-midi)

Ce carnet construit les tables de référence du projet avec l'**API DataFrame** :

1. du RDD au DataFrame : **schémas inférés** contre schéma explicite ;
2. lecture typée et **nettoyage** des relevés (valeurs nulles, outliers, types, `taux_occupation`) ;
3. **référentiel des stations** construit en Spark : identifiant, arrondissement, homonymes ;
4. table **`disponibilite`** au schéma cible ;
5. table **`meteo_horaire`** et table horaire de **qualité de l'air** (NO2, PM2.5) ;
6. **jointures broadcast** et colonnes calendaires ;
7. **mesure avec et sans cache** de la jointure répétée (contrainte n°6), lue dans le Spark UI ;
8. écriture en **Parquet partitionné** par année et par mois.

Il ne lit que les fichiers bruts de `data/` et la sortie JSON du carnet 01 n'est pas nécessaire.

> **ADAPTATIONS.**
> - L'énoncé prévoit des fichiers Parquet distribués par l'équipe pédagogique : l'historique public est
>   en CSV gzip (voir le README). On compare donc les schémas du RDD, du CSV inféré et du schéma
>   explicite, puis du Parquet **que l'on écrit** en fin de carnet.
> - La jointure temporelle se fait à **l'heure la plus proche** et non au quart d'heure : la météo et la
>   qualité de l'air sont horaires.

## 1. Session

Cellule commune à tous les carnets. On importe aussi les types et fonctions de `pyspark.sql`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..", "scripts").resolve()))
from config import *          # chemins, SEED, SHUFFLE_PARTITIONS, creer_session()

import csv
import shutil
import time
from datetime import datetime, timezone
from functools import reduce
from operator import or_

import pandas as pd
from pyspark import StorageLevel
from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = creer_session("ClimaCity-02-dataframe-pipeline")
sc = spark.sparkContext
print(f"Spark {spark.version} | {sc.defaultParallelism} coeurs | "
      f"shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')} | "
      f"AQE = {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Spark UI : {SPARK_UI}")

Spark 3.5.3 | 8 coeurs | shuffle.partitions = 8 | AQE = true
Spark UI : http://localhost:4040


## 2. Du RDD au DataFrame : trois façons d'obtenir un schéma

On compare trois DataFrames construits sur les mêmes données :

- **depuis un RDD** de tuples Python (parsing du carnet 01, sur le plus petit fichier) : `createDataFrame`
  déduit les types des objets Python ;
- **CSV avec `inferSchema=True`** : Spark lit **toutes** les données une première fois pour deviner les
  types, avant même la première action ;
- **CSV avec un schéma explicite** : aucune lecture préalable, et les types sont ceux qu'on a choisis.

In [2]:
MOTIF_VELIB = str(VELIB_RAW / "velib_*.csv.gz")
COLONNES_BRUTES = ["horodatage", "capacite", "velos_mecaniques", "velos_electriques",
                   "nom_station", "coordonnees", "en_service"]


def parser_releve(ligne: str):
    """Parse une ligne CSV brute (même logique que le carnet 01).

    Args:
        ligne: Ligne du fichier, coordonnées entre guillemets.

    Returns:
        Le tuple ``(horodatage, capacite, velos_mecaniques, velos_electriques, nom_station,
        coordonnees, en_service)`` typé, ou None si la ligne est inexploitable.

    Example:
        >>> parser_releve('2020-11-26T12:59Z,35,4,5,Nom,"48.86598,2.27572",True')[1]
        35
    """
    try:
        h, cap, meca, elec, nom, coord, service = next(csv.reader([ligne]))
        return (datetime.strptime(h, "%Y-%m-%dT%H:%MZ").replace(tzinfo=timezone.utc),
                int(cap), int(meca), int(elec), nom.strip(), coord, service == "True")
    except (ValueError, StopIteration):
        return None


petit = min(VELIB_RAW.glob("velib_*.csv.gz"), key=lambda f: f.stat().st_size)
sc.setJobDescription("Schéma : DataFrame depuis un RDD")
rdd = (sc.textFile(str(petit))
       .filter(lambda l: not l.startswith("horodatage,"))
       .map(parser_releve)
       .filter(lambda r: r is not None))
df_depuis_rdd = spark.createDataFrame(rdd, COLONNES_BRUTES)

sc.setJobDescription("Schéma : inferSchema sur les 10 fichiers")
t0 = time.perf_counter()
df_infere = spark.read.csv(MOTIF_VELIB, header=True, inferSchema=True)
t_inference = time.perf_counter() - t0

SCHEMA_BRUT = T.StructType([
    T.StructField("horodatage", T.TimestampType()),
    T.StructField("capacite", T.IntegerType()),
    T.StructField("velos_mecaniques", T.IntegerType()),
    T.StructField("velos_electriques", T.IntegerType()),
    T.StructField("nom_station", T.StringType()),
    T.StructField("coordonnees", T.StringType()),
    T.StructField("en_service", T.BooleanType()),
])
t0 = time.perf_counter()
df_explicite = (spark.read.schema(SCHEMA_BRUT)
                .option("header", True)
                .option("timestampFormat", "yyyy-MM-dd'T'HH:mmX")
                .csv(MOTIF_VELIB))
t_explicite = time.perf_counter() - t0

display(pd.DataFrame({
    "createDataFrame(RDD)": dict(df_depuis_rdd.dtypes),
    "CSV inferSchema": dict(df_infere.dtypes),
    "CSV schéma explicite": dict(df_explicite.dtypes),
}))
print(f"Définition du DataFrame : {t_inference:.1f} s avec inferSchema, {t_explicite:.2f} s avec le schéma")

,createDataFrame(RDD),CSV inferSchema,CSV schéma explicite
horodatage,timestamp,string,timestamp
capacite,bigint,int,int
velos_mecaniques,bigint,int,int
velos_electriques,bigint,int,int
nom_station,string,string,string
coordonnees,string,string,string
en_service,boolean,boolean,boolean


Définition du DataFrame : 3.8 s avec inferSchema, 0.16 s avec le schéma


Les trois schémas diffèrent :

- depuis le RDD, les entiers Python deviennent des **`bigint`** (64 bits) : Python n'a qu'un type entier,
  Spark choisit le plus large. Les `datetime` deviennent des `timestamp` ;
- `inferSchema` trouve des `int` et un `boolean`, mais doit **relire les 230 Mo** pour cela (un job
  complet avant toute action) ; c'est un coût payé à chaque définition du DataFrame ;
- le schéma explicite donne directement les **types du schéma cible** (`IntegerType`), sans lecture
  préalable. C'est lui qu'on utilise. Les coordonnées restent une chaîne, découpée juste après.

## 3. Lecture typée des relevés

Le lecteur CSV des DataFrames gère les guillemets : `coordonnees` arrive en un seul champ, qu'on découpe
avec `split`. Le nom est débarrassé de ses espaces superflues (le carnet 01 a montré trois noms
dédoublés par une espace finale). Tout reste paresseux : rien n'est lu à ce stade.

In [3]:
releves = (
    df_explicite
    .withColumn("nom_station", F.trim("nom_station"))
    .withColumn("latitude", F.split("coordonnees", ",").getItem(0).cast("double"))
    .withColumn("longitude", F.split("coordonnees", ",").getItem(1).cast("double"))
    .drop("coordonnees")
)
releves.printSchema()

root
 |-- horodatage: timestamp (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_mecaniques: integer (nullable = true)
 |-- velos_electriques: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- en_service: boolean (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



## 4. Nettoyage

Chaque relevé reçoit au plus un **motif de rejet**, évalué dans l'ordre ci-dessous ; un relevé sans
motif est conservé :

| Motif | Règle | Raison |
|---|---|---|
| `champ_nul` | une colonne nulle (y compris horodatage non parsé) | relevé inexploitable |
| `coordonnees_hors_zone` | hors de la boîte 48,6-49,1 N / 2,0-2,7 E | position fausse (« Arago - Paul Lafargue ») |
| `hors_service` | `en_service = False` | station fermée, compteurs sans rapport avec l'usage |
| `capacite_nulle_ou_negative` | `capacite <= 0` | taux d'occupation non défini |
| `velos_negatifs` | un compteur < 0 | impossible |

Les relevés avec **plus de vélos que de capacité** sont gardés mais **corrigés** : `bornettes_libres` est
bornée à 0, donc `taux_occupation` à 1. Une seule agrégation calcule le bilan de toutes les règles et les
quantiles de capacité (pour repérer d'éventuelles capacités aberrantes).

In [4]:
DANS_ZONE = F.col("latitude").between(48.6, 49.1) & F.col("longitude").between(2.0, 2.7)
MOTIFS = ["champ_nul", "coordonnees_hors_zone", "hors_service", "capacite_nulle_ou_negative",
          "velos_negatifs"]
une_colonne_nulle = reduce(or_, [F.col(c).isNull() for c in releves.columns])
motif_rejet = (
    F.when(une_colonne_nulle, "champ_nul")
    .when(~DANS_ZONE, "coordonnees_hors_zone")
    .when(~F.col("en_service"), "hors_service")
    .when(F.col("capacite") <= 0, "capacite_nulle_ou_negative")
    .when((F.col("velos_mecaniques") < 0) | (F.col("velos_electriques") < 0), "velos_negatifs")
)
releves_motif = releves.withColumn("motif_rejet", motif_rejet)
trop_de_velos = (F.col("velos_mecaniques") + F.col("velos_electriques")) > F.col("capacite")

sc.setJobDescription("Nettoyage : bilan des règles")
bilan = releves_motif.agg(
    F.count("*").alias("total"),
    *[F.sum((F.col("motif_rejet") == m).cast("int")).alias(m) for m in MOTIFS],
    F.sum((F.col("motif_rejet").isNull() & trop_de_velos).cast("int")).alias("corriges_velos_sup_capacite"),
    F.expr("percentile_approx(capacite, array(0.01, 0.5, 0.99))").alias("capacite_q01_q50_q99"),
    F.max("capacite").alias("capacite_max"),
).first().asDict()

n_brut = bilan["total"]
n_rejets = sum(bilan[m] for m in MOTIFS)
n_propres = n_brut - n_rejets
for cle, valeur in bilan.items():
    print(f"{cle:<30} {valeur}")
print(f"\nConservés : {n_propres:,} relevés sur {n_brut:,} ({n_propres / n_brut:.2%})".replace(",", " "))

total                          10986730
champ_nul                      0
coordonnees_hors_zone          680
hors_service                   216786
capacite_nulle_ou_negative     2
velos_negatifs                 0
corriges_velos_sup_capacite    2005
capacite_q01_q50_q99           [12, 30, 66]
capacite_max                   74

Conservés : 10 769 262 relevés sur 10 986 730 (98.02%)


Bilan : **98 % des relevés sont conservés**.

- **216 786 relevés hors service** (2 %), de loin la première cause de rejet. Ils contiennent presque toutes les capacités nulles vues au carnet 01 (64 762) : une fois les stations fermées écartées, il ne reste que 2 relevés de capacité nulle ;
- **680 relevés aux coordonnées fausses** (« Arago - Paul Lafargue », longitude égale à la latitude). Le référentiel GBFS contient une « Jean Jaurès - Paul Lafargue » à cette latitude (48,885), sans doute la même station : plutôt que de deviner une longitude, on écarte ces 680 relevés (0,006 %) ;
- aucun champ nul ni compteur négatif ;
- **2 005 relevés** ont plus de vélos que de capacité : ils sont gardés avec un taux d'occupation de 1.

Les capacités sont plausibles (de 12 à 66 places pour 98 % des relevés, 74 au maximum) : pas d'outlier à corriger de ce côté.

On applique les règles et on calcule les deux colonnes dérivées du schéma cible :
`bornettes_libres = max(0, capacite - velos)` et `taux_occupation = (capacite - bornettes_libres) / capacite`.

In [5]:
propres = (
    releves_motif
    .filter(F.col("motif_rejet").isNull())
    .drop("motif_rejet", "en_service")
    .withColumn("bornettes_libres",
                F.greatest(F.lit(0), F.col("capacite") - F.col("velos_mecaniques") - F.col("velos_electriques")))
    .withColumn("taux_occupation",
                (F.col("capacite") - F.col("bornettes_libres")) / F.col("capacite"))
)
propres.select("capacite", "velos_mecaniques", "velos_electriques", "bornettes_libres",
               "taux_occupation").limit(5).show()

+--------+----------------+-----------------+----------------+-------------------+
|capacite|velos_mecaniques|velos_electriques|bornettes_libres|    taux_occupation|
+--------+----------------+-----------------+----------------+-------------------+
|      35|               3|                2|              30|0.14285714285714285|
|      55|              47|                5|               3| 0.9454545454545454|
|      20|               2|                1|              17|               0.15|
|      21|               2|                0|              19|0.09523809523809523|
|      30|              19|                5|               6|                0.8|
+--------+----------------+-----------------+----------------+-------------------+



## 5. Référentiel des stations, construit en Spark

L'historique n'a pas d'identifiant de station. On part des **positions** (couples nom, coordonnées) et on
les rattache au référentiel GBFS actuel (1 520 stations, `stationCode`) en trois étapes :

1. **par le nom** : la station GBFS la plus proche, à moins de 300 m, dont le nom normalisé (minuscules,
   sans accents ni ponctuation) est similaire à au moins 80 % (distance de Levenshtein). Cela absorbe les
   fautes de frappe (« Champerret » / « Champeret ») et regroupe les positions d'une station déplacée ;
2. **par la position** : pour les positions restantes, la station GBFS la plus proche à moins de 50 m,
   si elle n'est pas déjà prise (station renommée depuis 2021) ; une station GBFS ne sert qu'une fois ;
3. **identifiant de substitution** (100 001, 100 002...) : stations fermées depuis 2021, absentes du
   référentiel actuel.

Les **homonymes** (« Place de l'Europe » à Paris et à Alfortville) sont départagés naturellement : leurs
positions sont à plusieurs kilomètres, et chacune ne trouve que sa propre station. L'**arrondissement**
vient toujours de la station GBFS la plus proche (`stationCode // 1000` : 1 à 20 pour Paris, un code de
commune au-delà), même pour les stations fermées : c'est une information géographique.

Le produit cartésien positions × GBFS (environ 2 millions de paires) reste petit : on diffuse le côté GBFS
(`broadcast`) et tout se calcule dans la JVM.

In [6]:
def distance_m(lat1: Column, lon1: Column, lat2: Column, lon2: Column) -> Column:
    """Expression Spark de la distance à vol d'oiseau (formule de haversine).

    Args:
        lat1: Colonne de latitude du premier point, en degrés.
        lon1: Colonne de longitude du premier point, en degrés.
        lat2: Colonne de latitude du second point, en degrés.
        lon2: Colonne de longitude du second point, en degrés.

    Returns:
        Une colonne ``double`` : la distance en mètres.

    Example:
        >>> df.withColumn("d", distance_m(F.col("lat"), F.col("lon"), F.lit(48.8584), F.lit(2.2945)))
    """
    a = (F.sin(F.radians(lat2 - lat1) / 2) ** 2
         + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(F.radians(lon2 - lon1) / 2) ** 2)
    return 2 * 6_371_000 * F.asin(F.sqrt(a))


def normaliser_nom(nom: Column) -> Column:
    """Normalise un nom de station pour la comparaison : minuscules, sans accents ni ponctuation.

    Args:
        nom: Colonne de noms.

    Returns:
        Une colonne de chaînes réduites aux lettres et chiffres non accentués.

    Example:
        >>> spark.range(1).select(normaliser_nom(F.lit("Place de l'Édit de Nantes"))).first()[0]
        'placededitdenantes'
    """
    sans_accents = F.translate(F.lower(nom), "àâäéèêëîïôöùûüçœ", "aaaeeeeiioouuuco")
    return F.regexp_replace(sans_accents, "[^a-z0-9]", "")


CLE = ["nom_station", "latitude", "longitude"]

sc.setJobDescription("Référentiel : positions des stations")
positions = (propres.groupBy(*CLE)
             .agg(F.count("*").alias("releves"), F.min("horodatage").alias("premier"),
                  F.max("horodatage").alias("dernier"))
             .persist())
gbfs = (spark.read.json(str(GBFS_JSON))
        .select(F.explode("data.stations").alias("s"))
        .select(F.col("s.stationCode").cast("int").alias("code_gbfs"), F.col("s.name").alias("nom_gbfs"),
                F.col("s.lat").alias("lat_gbfs"), F.col("s.lon").alias("lon_gbfs")))

n1, n2 = normaliser_nom(F.col("nom_station")), normaliser_nom(F.col("nom_gbfs"))
candidats = (
    positions.select(*CLE).crossJoin(F.broadcast(gbfs))
    .withColumn("distance_m", distance_m(F.col("latitude"), F.col("longitude"),
                                         F.col("lat_gbfs"), F.col("lon_gbfs")))
    .withColumn("similarite_nom", 1 - F.levenshtein(n1, n2) / F.greatest(F.length(n1), F.length(n2)))
)
par_position = Window.partitionBy(*CLE).orderBy("distance_m", "code_gbfs")


def plus_proche(df: DataFrame) -> DataFrame:
    """Garde, pour chaque position, le candidat GBFS le plus proche.

    Args:
        df: Candidats (une ligne par couple position / station GBFS).

    Returns:
        Une ligne par position.

    Example:
        >>> plus_proche(candidats.filter("distance_m <= 50"))
    """
    return df.withColumn("rang", F.row_number().over(par_position)).filter("rang = 1").drop("rang")


geographie = plus_proche(candidats).select(
    *CLE, (F.col("code_gbfs") / 1000).cast("int").alias("arrondissement"),
    F.col("distance_m").alias("distance_gbfs_plus_proche_m"))

etape_nom = (plus_proche(candidats.filter("distance_m <= 300 AND similarite_nom >= 0.8"))
             .withColumn("rattachement", F.lit("nom")))
restantes = positions.select(*CLE).join(etape_nom, CLE, "left_anti")
etape_position = (
    plus_proche(candidats.join(restantes, CLE)
                .join(etape_nom.select("code_gbfs"), "code_gbfs", "left_anti")
                .filter("distance_m <= 50"))
    .withColumn("rang", F.row_number().over(Window.partitionBy("code_gbfs").orderBy("distance_m")))
    .filter("rang = 1").drop("rang")
    .withColumn("rattachement", F.lit("position"))
)
rattachees = etape_nom.unionByName(etape_position).select(
    *CLE, "code_gbfs", "nom_gbfs", "distance_m", "similarite_nom", "rattachement")

sans_code = F.col("code_gbfs").isNull()
stations_calcul = (
    positions.join(rattachees, CLE, "left").join(geographie, CLE)
    .withColumn("rattachement", F.coalesce("rattachement", F.lit("substitution")))
    .withColumn("rang_substitution",
                F.row_number().over(Window.partitionBy(sans_code).orderBy(*CLE)))
    .withColumn("station_id",
                F.when(sans_code, 100_000 + F.col("rang_substitution")).otherwise(F.col("code_gbfs"))
                .cast("int"))
    .drop("rang_substitution")
)

sc.setJobDescription("Référentiel : écriture")
stations_calcul.coalesce(1).write.mode("overwrite").parquet(str(STATIONS_PARQUET))
positions.unpersist()
stations = spark.read.parquet(str(STATIONS_PARQUET))

display(stations.groupBy("rattachement")
        .agg(F.count("*").alias("positions"), F.countDistinct("station_id").alias("stations"),
             F.sum("releves").alias("releves"), F.round(F.max("distance_m"), 1).alias("distance_max_m"),
             F.round(F.min("similarite_nom"), 2).alias("similarite_min"))
        .orderBy("rattachement").toPandas())
print(f"{stations.count()} positions -> {stations.select('station_id').distinct().count()} stations")

,rattachement,positions,stations,releves,distance_max_m,similarite_min
0,nom,1301,1299,10152657,136.0,0.91
1,position,36,36,279011,48.4,0.14
2,substitution,47,47,337594,NaN,NaN


1384 positions -> 1382 stations


Contrôle sur les cas difficiles repérés au carnet 01 : les homonymes reçoivent des identifiants
différents, les positions d'une station déplacée le même identifiant.

In [7]:
noms_doubles = stations.groupBy("nom_station").count().filter("count > 1").select("nom_station")
display(stations.join(noms_doubles, "nom_station")
        .select("nom_station", "latitude", "longitude", "station_id", "arrondissement", "rattachement",
                "nom_gbfs", F.round("distance_m", 1).alias("distance_m"), "releves")
        .orderBy("nom_station", "premier").toPandas())

print("Exemples de rattachements par la position (stations renommées depuis 2021) :")
display(stations.filter("rattachement = 'position'")
        .select("nom_station", "nom_gbfs", F.round("distance_m", 1).alias("distance_m"))
        .orderBy("distance_m").limit(8).toPandas())
print("Stations sans équivalent actuel (identifiant de substitution) :")
display(stations.filter("rattachement = 'substitution'")
        .select("station_id", "nom_station", "arrondissement",
                F.round("distance_gbfs_plus_proche_m").alias("gbfs_le_plus_proche_m"), "releves")
        .orderBy("station_id").limit(10).toPandas())

,nom_station,latitude,longitude,station_id,arrondissement,rattachement,nom_gbfs,distance_m,releves
0,Château - République,48.86292,2.41550,31706,31,nom,Château - République,0.5,7866
1,Château - République,48.90593,2.25363,22508,22,nom,Château -République,13.3,7866
2,Neuilly - Petit,48.89929,2.29669,21118,21,nom,Neuilly - Petit,46.9,2294
3,Neuilly - Petit,48.89897,2.29628,21118,21,nom,Neuilly - Petit,0.5,5572
4,Place Nelson Mandela,48.86209,2.19658,25006,25,nom,Place Nelson Mandela,50.5,7866
5,Place Nelson Mandela,48.89787,2.21828,92006,92,nom,Place Nelson Mandela,0.5,7866
6,Place de l'Eglise,48.79192,2.39506,44004,44,position,Place de l'Eglise - Vitry-sur-Seine,0.4,7866
7,Place de l'Eglise,48.89281,2.41244,35008,35,position,Place de l'Eglise - Pantin,0.4,7866
8,Place de l'Europe,48.78914,2.42848,41405,41,position,Place de l'Europe - Alfortville,0.3,7866
9,Place de l'Europe,48.82530,2.39457,42209,42,position,Place de l'Europe - Quai de Bercy,0.2,7866


Exemples de rattachements par la position (stations renommées depuis 2021) :


,nom_station,nom_gbfs,distance_m
0,Cimetière Communal,Cimetière d'Issy-les-Moulineaux,0.1
1,Place de l'Europe,Place de l'Europe - Quai de Bercy,0.2
2,Cimetière,Cimetière de Fontenay-sous-Bois,0.3
3,Place de l'Europe,Place de l'Europe - Alfortville,0.3
4,Place de l'Eglise,Place de l'Eglise - Vitry-sur-Seine,0.4
5,Ney - Charles Hermite,Station Paris Basketball - Ney - Charles Hermite,0.4
6,Place de l'Eglise,Place de l'Eglise - Pantin,0.4
7,Stade de France,Stade de France - Francis de Pressensé,0.4


Stations sans équivalent actuel (identifiant de substitution) :


,station_id,nom_station,arrondissement,gbfs_le_plus_proche_m,releves
0,100001,Alésia - Suisses,14,98.0,7866
1,100002,Bagnolet - Orteaux,20,102.0,7866
2,100003,Berges de Seine - Musée d'Orsay,60,273.0,3093
3,100004,Bibliothèque Nationale,2,132.0,7866
4,100005,Charenton - Jardiniers,12,195.0,7866
5,100006,Charles de Gaulle - Graviers,22,287.0,7866
6,100007,Charles de Gaulle - Madrid,22,138.0,7866
7,100008,Chateaubriand - Friedland,8,65.0,7866
8,100009,Château - Place de Catalogne,14,95.0,7866
9,100010,Croix Nivert - Square Saint-Lambert,15,174.0,7866


Sur les 1 384 positions qui ont au moins un relevé conservé :

- **1 301 sont rattachées par le nom** (94 % des relevés), toutes à moins de 140 m, avec une similarité d'au moins 0,91 ;
- **36 par la position**, à moins de 50 m : des stations renommées depuis 2021, souvent avec le nom de la commune ajouté (« Place de l'Europe - Alfortville ») ;
- **47 reçoivent un identifiant de substitution** (3 % des relevés) : stations fermées, ou déplacées de plus de 50 m, depuis 2021.

Les homonymes sont bien séparés : chaque « Château - République », « Place de l'Europe », « Place de l'Eglise », « Place Nelson Mandela » ou « Université » a son propre identifiant, dans sa commune. Les deux fusions (1 384 positions pour 1 382 stations) sont attendues : Neuilly - Petit (déplacée de 47 m le 29/12) et Parmentier - Abel rabaud / Rabaud (renommée le 03/12). Les autres déplacements vus au carnet 01 (Chabanais, Parc Floral du Bel Air, Hôpital Européen Georges Pompidou) n'ont plus qu'une position, ou aucune, une fois les relevés hors service retirés : le Parc Floral, par exemple, est resté fermé tout l'hiver.

## 6. Table `disponibilite`

On attache l'identifiant et l'arrondissement à chaque relevé par une **jointure broadcast** sur la
position (1 400 lignes diffusées à toutes les tâches, aucun shuffle des 11 millions de relevés).

Deux positions d'une même station peuvent coexister au même horodatage (pendant un déplacement). Pour
garantir l'unicité de `(station_id, horodatage)`, on garde le relevé de la position la plus suivie, avec
une fonction de fenêtre (`row_number`). Puis on sélectionne les colonnes du **schéma cible**, avec ses
types exacts.

In [8]:
SCHEMA_DISPONIBILITE = T.StructType([
    T.StructField("station_id", T.IntegerType()),
    T.StructField("nom_station", T.StringType()),
    T.StructField("arrondissement", T.IntegerType()),
    T.StructField("latitude", T.DoubleType()),
    T.StructField("longitude", T.DoubleType()),
    T.StructField("horodatage", T.TimestampType()),
    T.StructField("velos_mecaniques", T.IntegerType()),
    T.StructField("velos_electriques", T.IntegerType()),
    T.StructField("bornettes_libres", T.IntegerType()),
    T.StructField("capacite_totale", T.IntegerType()),
    T.StructField("taux_occupation", T.DoubleType()),
])

doublons = Window.partitionBy("station_id", "horodatage").orderBy(F.desc("releves"), "nom_station")
disponibilite = (
    propres.withColumnRenamed("capacite", "capacite_totale")
    .join(F.broadcast(stations.select(*CLE, "station_id", "arrondissement", "releves")), CLE)
    .withColumn("rang", F.row_number().over(doublons))
    .filter("rang = 1")
    .select([F.col(c.name).cast(c.dataType) for c in SCHEMA_DISPONIBILITE])
)
disponibilite.printSchema()
assert [(c.name, c.dataType) for c in disponibilite.schema] == \
       [(c.name, c.dataType) for c in SCHEMA_DISPONIBILITE], "schéma cible non respecté"

root
 |-- station_id: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- arrondissement: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_mecaniques: integer (nullable = true)
 |-- velos_electriques: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = false)
 |-- capacite_totale: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)



## 7. Table `meteo_horaire`

Le CSV Open-Meteo est déjà en UTC et ne compte que 3 240 lignes. Schéma explicite, renommage vers le
schéma cible, et `est_pluie = precipitation_mm > 0,5`. On vérifie qu'il ne manque aucune heure.

In [9]:
SCHEMA_METEO_CSV = T.StructType([
    T.StructField("time", T.TimestampType()),
    T.StructField("temperature_2m", T.DoubleType()),
    T.StructField("relative_humidity_2m", T.DoubleType()),
    T.StructField("precipitation", T.DoubleType()),
    T.StructField("wind_speed_10m", T.DoubleType()),
    T.StructField("cloud_cover", T.DoubleType()),
    T.StructField("weather_code", T.IntegerType()),
])
meteo = (
    spark.read.schema(SCHEMA_METEO_CSV).option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm")
    .csv(str(METEO_CSV))
    .select(F.col("time").alias("horodatage"),
            F.col("temperature_2m").alias("temperature_c"),
            F.col("relative_humidity_2m").alias("humidite_pct"),
            F.col("precipitation").alias("precipitation_mm"),
            F.col("wind_speed_10m").alias("vitesse_vent_kmh"),
            (F.col("precipitation") > 0.5).alias("est_pluie"))
)

sc.setJobDescription("Météo : contrôles")
controle = meteo.agg(
    F.count("*").alias("heures"), F.countDistinct("horodatage").alias("distinctes"),
    F.min("horodatage").alias("debut"), F.max("horodatage").alias("fin"),
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"nuls_{c}") for c in meteo.columns],
    F.round(F.avg(F.col("est_pluie").cast("int")), 4).alias("part_heures_pluie"),
    F.round(F.min("temperature_c"), 1).alias("t_min"), F.round(F.max("temperature_c"), 1).alias("t_max"),
).first().asDict()
attendues = int((controle["fin"] - controle["debut"]).total_seconds() // 3600) + 1
for cle, valeur in controle.items():
    print(f"{cle:<26} {valeur}")
assert controle["heures"] == controle["distinctes"] == attendues, "heures manquantes ou en double"

DEBUT, FIN = controle["debut"], controle["fin"]
meteo.coalesce(1).write.mode("overwrite").parquet(str(METEO_PARQUET))

heures                     3240
distinctes                 3240
debut                      2020-11-26 00:00:00
fin                        2021-04-09 23:00:00
nuls_horodatage            0
nuls_temperature_c         0
nuls_humidite_pct          0
nuls_precipitation_mm      0
nuls_vitesse_vent_kmh      0
nuls_est_pluie             0
part_heures_pluie          0.0571
t_min                      -6.0
t_max                      23.5


La table est complète : 3 240 heures consécutives du 26/11/2020 00:00 au 09/04/2021 23:00 UTC, sans valeur nulle. Il a plu plus de 0,5 mm pendant **5,7 % des heures** (185 heures) ; les températures vont de -6 °C à 23,5 °C. `est_pluie` est donc un indicateur rare, ce qu'il faudra garder en tête pour la question SQL sur la pluie et pour le modèle.

## 8. Qualité de l'air horaire (NO2, PM2.5)

Les fichiers de l'EEA couvrent 2013 à 2025 pour 37 points de mesure du NO2 (code de polluant 8) et 13 du
PM2.5 (6001). On ne garde que les valeurs **valides** (`Validity > 0`, les invalides valent -9999).

**Fuseau horaire.** La documentation de l'EEA indique des heures **locales sans heure d'été** (UTC+1 pour
la France), mais on le vérifie sur les données. Un test interne d'abord : si les horodatages suivaient
l'heure d'été, le pic du matin aux heures de pointe resterait à la même heure toute l'année ; s'ils sont
à décalage fixe, il **avance d'une heure en été** dans ce repère. Puis un test externe, avec un repère
dont on connaît le fuseau : l'**usage des Vélib'** (en UTC), mesuré par le nombre de vélos absents des
stations, a la même forme horaire que le trafic, et donc que le NO2 des stations de trafic.

In [10]:
air_valide = (spark.read.parquet(str(AIR_RAW))
              .filter((F.col("Validity") > 0) & F.col("Pollutant").isin(8, 6001))
              .withColumn("valeur", F.col("Value").cast("double")))
jour_ouvre = F.dayofweek("Start").between(2, 6)

sc.setJobDescription("Air : profil horaire du NO2 par saison")
profil_saison = (
    air_valide.filter((F.col("Pollutant") == 8) & jour_ouvre)
    .withColumn("saison", F.when(F.month("Start").isin(12, 1, 2), "hiver")
                .when(F.month("Start").isin(6, 7, 8), "été"))
    .filter(F.col("saison").isNotNull())
    .groupBy("saison", F.hour("Start").alias("heure_fichier"))
    .agg(F.avg("valeur").alias("no2"))
    .toPandas()
    .pivot(index="heure_fichier", columns="saison", values="no2")
)
matin = profil_saison.loc[4:11]
print("Pic du matin dans le repère des fichiers (heure de début de mesure, jours ouvrés, 2013-2025) :")
print(f"  hiver : {matin['hiver'].idxmax()} h | été : {matin['été'].idxmax()} h")

Pic du matin dans le repère des fichiers (heure de début de mesure, jours ouvrés, 2013-2025) :
  hiver : 8 h | été : 7 h


Le pic avance en été : les horodatages ont un **décalage fixe**, sans heure d'été. Reste à savoir lequel,
UTC ou UTC+1. On construit le profil d'usage des Vélib' : pour chaque passage du collecteur, le nombre de
vélos **absents** (capacité - vélos présents, sommé sur les stations, rapporté à sa moyenne du jour),
moyenné par heure UTC sur les jours ouvrés de l'hiver. On le corrèle au profil horaire du NO2 de la même
période, pour un décalage de -3 h à +2 h appliqué aux heures des fichiers EEA.

In [11]:
sc.setJobDescription("Air : profil d'usage Vélib' en UTC")
usage = (
    disponibilite
    .groupBy("horodatage")
    .agg(F.sum(F.col("capacite_totale") - F.col("velos_mecaniques") - F.col("velos_electriques"))
         .alias("velos_absents"))
    .withColumn("jour", F.to_date("horodatage"))
    .withColumn("relatif", F.col("velos_absents") / F.avg("velos_absents").over(Window.partitionBy("jour")))
    .filter(F.dayofweek("horodatage").between(2, 6))
    .groupBy(F.hour("horodatage").alias("heure_utc"))
    .agg(F.avg("relatif").alias("usage_velib"))
    .toPandas().set_index("heure_utc").sort_index()["usage_velib"]
)

sc.setJobDescription("Air : profil horaire du NO2 sur la période")
no2_periode = (
    air_valide.filter((F.col("Pollutant") == 8) & jour_ouvre
                      & F.col("Start").between(DEBUT, FIN))
    .groupBy(F.hour("Start").alias("heure_fichier"))
    .agg(F.avg("valeur").alias("no2"))
    .toPandas().set_index("heure_fichier").sort_index()["no2"]
)

correlations = {}
for decalage in range(-3, 3):
    # heure UTC = heure du fichier + décalage
    decale = pd.Series(no2_periode.values, index=(no2_periode.index + decalage) % 24).sort_index()
    correlations[decalage] = usage.corr(decale)
display(pd.Series(correlations, name="corrélation usage Vélib' / NO2").rename_axis("décalage (h)").round(3)
        .to_frame().T)
meilleur = max(correlations, key=correlations.get)
print(f"Meilleur décalage : {meilleur:+d} h -> heure UTC = heure des fichiers {meilleur:+d} h")

décalage (h),-3,-2,-1,0,1,2
corrélation usage Vélib' / NO2,0.207,0.392,0.525,0.565,0.502,0.381


Meilleur décalage : +0 h -> heure UTC = heure des fichiers +0 h


Le profil d'usage des Vélib' corrèle le mieux avec le NO2 **sans décalage** (0,565), mais à peine plus qu'à -1 h (0,525). Ce test seul ne suffit pas : le NO2 **retarde physiquement** sur le trafic (il s'accumule pendant et après le pic), ce qui favorise les décalages proches de 0 même si le fichier est en UTC+1.

Second repère externe, sans lien avec le trafic : le **vent** disperse le NO2 et la météo Open-Meteo est
en UTC. Pour ne pas retrouver simplement le cycle journalier commun aux deux séries, on retire à chacune
son profil moyen par heure et par type de jour (ouvré / week-end), puis on corrèle les **anomalies**
de NO2 (heure de début dans le fichier) à celles du vent `k` heures plus tard ou plus tôt. Les deux séries
font 3 240 heures : on les ramène en Pandas pour ce calcul.

In [12]:
def anomalies_horaires(serie: pd.Series) -> pd.Series:
    """Retire d'une série horaire son profil moyen par heure et par type de jour.

    Args:
        serie: Série indexée par des horodatages horaires.

    Returns:
        La série des écarts au profil (même index).

    Example:
        >>> anomalies_horaires(vent).groupby(vent.index.hour).mean().abs().max() < 1e-9
        True
    """
    cles = [serie.index.hour, serie.index.dayofweek >= 5]
    return serie - serie.groupby(cles).transform("mean")


sc.setJobDescription("Air : NO2 horaire moyen sur la période")
no2_heure = (air_valide.filter((F.col("Pollutant") == 8) & F.col("Start").between(DEBUT, FIN))
             .groupBy("Start").agg(F.avg("valeur").alias("no2"))
             .toPandas().set_index("Start").sort_index()["no2"])
vent = meteo.select("horodatage", "vitesse_vent_kmh").toPandas().set_index("horodatage")["vitesse_vent_kmh"]

a_no2, a_vent = anomalies_horaires(no2_heure), anomalies_horaires(vent)
correlations_vent = {}
for decalage in range(-3, 3):
    vent_decale = a_vent.copy()
    vent_decale.index = vent_decale.index - pd.Timedelta(hours=decalage)   # vent à t + décalage
    correlations_vent[decalage] = a_no2.corr(vent_decale)
display(pd.Series(correlations_vent, name="corrélation anomalies NO2 / vent").rename_axis("décalage (h)")
        .round(3).to_frame().T)

décalage (h),-3,-2,-1,0,1,2
corrélation anomalies NO2 / vent,-0.593,-0.602,-0.601,-0.585,-0.562,-0.539


Bilan des trois tests :

- le pic du matin avance d'une heure en été : les horodatages ont un **décalage fixe**, sans heure d'été. C'est le seul résultat net ;
- usage Vélib' : corrélation maximale à 0 h, à peine meilleure qu'à -1 h ;
- vent : corrélation (négative) maximale entre -2 h et -1 h, avec une courbe très plate.

Les repères externes ne départagent pas UTC et UTC+1 à eux seuls, car le NO2 réagit avec retard au trafic comme au vent. En revanche, ils sont **cohérents entre eux sous l'hypothèse UTC+1** : le NO2 suit alors le trafic et le vent avec le même retard, d'environ une heure. En UTC, il faudrait un retard nul sur le trafic, mais d'une à deux heures sur le vent. On retient donc **UTC+1**, la convention des données européennes (heure locale sans heure d'été), et on décale d'une heure vers l'UTC.

L'incertitude résiduelle est d'**une heure sur une variable horaire du bonus** : elle ne touche ni la disponibilité ni la météo.

On construit la table horaire : décalage d'une heure vers l'UTC, puis moyenne sur les points de mesure
de chaque polluant. Chaque valeur est **étiquetée par la fin de son intervalle** (`End`), comme la
précipitation d'Open-Meteo (cumul de l'heure écoulée) : l'étiquette 13:00 couvre 12:00-13:00.

In [13]:
DECALAGE_EEA = F.expr("INTERVAL 1 HOUR")
est_no2, est_pm25 = F.col("Pollutant") == 8, F.col("Pollutant") == 6001
air_horaire = (
    air_valide
    .withColumn("horodatage", F.col("End") - DECALAGE_EEA)
    .filter(F.col("horodatage").between(DEBUT, FIN))
    .groupBy("horodatage")
    .agg(F.avg(F.when(est_no2, F.col("valeur"))).alias("no2_ugm3"),
         F.avg(F.when(est_pm25, F.col("valeur"))).alias("pm25_ugm3"),
         F.countDistinct(F.when(est_no2, F.col("Samplingpoint"))).alias("points_no2"),
         F.countDistinct(F.when(est_pm25, F.col("Samplingpoint"))).alias("points_pm25"))
)

sc.setJobDescription("Air : écriture de la table horaire")
air_horaire.coalesce(1).write.mode("overwrite").parquet(str(AIR_PARQUET))
air_horaire = spark.read.parquet(str(AIR_PARQUET))
completude = air_horaire.agg(
    F.count("*").alias("heures"), F.count("no2_ugm3").alias("heures_no2"),
    F.count("pm25_ugm3").alias("heures_pm25"),
    F.round(F.avg("points_no2"), 1).alias("points_no2_moyen"),
    F.round(F.avg("points_pm25"), 1).alias("points_pm25_moyen"),
    F.round(F.avg("no2_ugm3"), 1).alias("no2_moyen"), F.round(F.avg("pm25_ugm3"), 1).alias("pm25_moyen"),
).first().asDict()
print(f"{attendues} heures dans la période")
for cle, valeur in completude.items():
    print(f"{cle:<18} {valeur}")

3240 heures dans la période
heures             3240
heures_no2         3240
heures_pm25        3240
points_no2_moyen   31.3
points_pm25_moyen  7.0
no2_moyen          33.1
pm25_moyen         15.4


## 9. Jointures broadcast et colonnes calendaires

La table consolidée part de `disponibilite` et lui ajoute :

- `horodatage_heure` : l'heure UTC **la plus proche** du relevé (12:40 -> 13:00), clé de jointure ;
- des colonnes calendaires à l'**heure de Paris** (`date`, `heure`, `jour_semaine` de 1 = lundi à
  7 = dimanche, `est_weekend`), car les usages suivent l'heure locale ; `annee` et `mois` servent au
  partitionnement. Les horodatages restent en UTC ;
- la météo et la qualité de l'air, par deux **jointures broadcast** à gauche : les tables horaires
  (3 240 lignes) sont copiées dans chaque tâche, et les 11 millions de relevés ne sont pas redistribués.

Le plan physique confirme le choix de Spark : `BroadcastHashJoin`, sans `Exchange` (shuffle) côté
relevés pour ces deux jointures.

In [14]:
def ajouter_calendrier(df: DataFrame) -> DataFrame:
    """Ajoute la clé horaire de jointure et les colonnes calendaires à l'heure de Paris.

    Args:
        df: DataFrame avec une colonne ``horodatage`` en UTC.

    Returns:
        Le DataFrame avec ``horodatage_heure`` (heure UTC la plus proche), ``date``, ``heure``,
        ``jour_semaine`` (1 = lundi), ``est_weekend``, ``annee`` et ``mois`` (heure de Paris).

    Example:
        >>> ajouter_calendrier(disponibilite).select("horodatage", "heure", "jour_semaine").first()
    """
    local = F.from_utc_timestamp("horodatage", "Europe/Paris")
    return (
        df.withColumn("horodatage_heure",
                      F.date_trunc("hour", F.col("horodatage") + F.expr("INTERVAL 30 MINUTES")))
        .withColumn("date", F.to_date(local))
        .withColumn("heure", F.hour(local))
        .withColumn("jour_semaine", (F.dayofweek(local) + 5) % 7 + 1)
        .withColumn("est_weekend", F.col("jour_semaine") >= 6)
        .withColumn("annee", F.year(local))
        .withColumn("mois", F.month(local))
    )


def consolider(dispo: DataFrame, meteo_h: DataFrame, air_h: DataFrame) -> DataFrame:
    """Joint relevés, météo et qualité de l'air sur l'heure la plus proche.

    Args:
        dispo: Table ``disponibilite``.
        meteo_h: Table ``meteo_horaire``.
        air_h: Table horaire de qualité de l'air.

    Returns:
        La table consolidée : relevés + calendrier + météo + NO2 / PM2.5.

    Example:
        >>> consolider(disponibilite, meteo, air_horaire).count()   # doctest: +SKIP
    """
    return (
        ajouter_calendrier(dispo)
        .join(F.broadcast(meteo_h.withColumnRenamed("horodatage", "horodatage_heure")),
              "horodatage_heure", "left")
        .join(F.broadcast(air_h.select(F.col("horodatage").alias("horodatage_heure"),
                                       "no2_ugm3", "pm25_ugm3")),
              "horodatage_heure", "left")
    )


consolide = consolider(disponibilite, meteo, air_horaire)
plan = consolide._jdf.queryExecution().executedPlan().toString()
print("\n".join(l for l in plan.splitlines() if any(m in l for m in ("Join", "Exchange", "Scan", "Window"))))

   +- BroadcastHashJoin [horodatage_heure#2682], [horodatage_heure#2824], LeftOuter, BuildRight, false
      :  +- BroadcastHashJoin [horodatage_heure#2682], [horodatage_heure#2794], LeftOuter, BuildRight, false
      :     :        +- Window [row_number() windowspecdefinition(station_id#1820, horodatage#45, releves#1810L DESC NULLS LAST, nom_station#59 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rang#2032], [station_id#1820, horodatage#45], [releves#1810L DESC NULLS LAST, nom_station#59 ASC NULLS FIRST]
      :     :           +- WindowGroupLimit [station_id#1820, horodatage#45], [releves#1810L DESC NULLS LAST, nom_station#59 ASC NULLS FIRST], row_number(), 1, Final
      :     :                 +- Exchange hashpartitioning(station_id#1820, horodatage#45, 8), ENSURE_REQUIREMENTS, [plan_id=6990]
      :     :                    +- WindowGroupLimit [station_id#1820, horodatage#45], [releves#1810L DESC NULLS LAST, nom_station#59 ASC NULLS FIR

Le plan contient deux `BroadcastHashJoin` (météo, air) et la jointure broadcast avec le référentiel ;
le seul `Exchange` sur les relevés est celui de la fenêtre de dédoublonnage (`Window`), qui doit regrouper
les relevés par `(station_id, horodatage)`.

## 10. Mesure avec et sans cache (contrainte n°6)

La même requête -- jointure avec la météo puis taux d'occupation moyen avec et sans pluie -- est lancée
**trois fois** dans trois configurations :

- **A, sans cache** : chaque exécution relit les gzip, nettoie, joint, dédoublonne ;
- **B, météo persistée** (`persist(MEMORY_AND_DISK)`, comme le propose l'énoncé) ;
- **C, météo et `disponibilite` persistées**.

La première exécution de B et C remplit le cache (son coût est mesuré à part). Chaque exécution est
nommée, et l'API REST du Spark UI donne pour chacune le volume lu en entrée et le shuffle.

In [15]:
def requete_jointure(dispo: DataFrame, meteo_h: DataFrame) -> list:
    """Requête de référence pour les mesures : jointure météo puis agrégation par pluie.

    Args:
        dispo: Table ``disponibilite`` (persistée ou non).
        meteo_h: Table ``meteo_horaire`` (persistée ou non).

    Returns:
        Les lignes ``(est_pluie, taux_moyen, releves)``.

    Example:
        >>> requete_jointure(disponibilite, meteo)   # doctest: +SKIP
    """
    return (consolider(dispo, meteo_h, air_horaire)
            .groupBy("est_pluie")
            .agg(F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"), F.count("*").alias("releves"))
            .orderBy("est_pluie")
            .collect())


def mesurer(scenario: str, repetitions: int = 3) -> list:
    """Chronomètre la requête de référence et relève ses stages dans le Spark UI.

    Args:
        scenario: Lettre du scénario, utilisée dans la description des jobs.
        repetitions: Nombre d'exécutions.

    Returns:
        Une ligne par exécution : scénario, essai, durée, entrée lue (Mo), shuffle écrit (Mo).

    Example:
        >>> mesurer("A", repetitions=1)   # doctest: +SKIP
    """
    lignes = []
    for essai in range(1, repetitions + 1):
        description = f"Cache {scenario} : essai {essai}"
        sc.setJobDescription(description)
        t0 = time.perf_counter()
        resultat = requete_jointure(disponibilite, meteo)
        duree = time.perf_counter() - t0
        stages = stages_spark_ui(spark, description, tous_les_jobs=True)
        lignes.append((scenario, essai, duree, sum(s["entree_mo"] for s in stages),
                       sum(s["shuffle_ecrit_ko"] for s in stages) / 1e3, len(stages)))
    print(f"Scénario {scenario} : {resultat}")
    return lignes


mesures_cache = mesurer("A")

meteo.persist(StorageLevel.MEMORY_AND_DISK)
sc.setJobDescription("Cache B : remplissage")
t0 = time.perf_counter()
meteo.count()
remplissage_meteo = time.perf_counter() - t0
mesures_cache += mesurer("B")

disponibilite.persist(StorageLevel.MEMORY_AND_DISK)
sc.setJobDescription("Cache C : remplissage")
t0 = time.perf_counter()
n_disponibilite = disponibilite.count()
remplissage_dispo = time.perf_counter() - t0
mesures_cache += mesurer("C")

tableau_cache = pd.DataFrame(mesures_cache, columns=["scenario", "essai", "duree_s", "entree_mo",
                                                     "shuffle_mo", "stages"])
display(tableau_cache.round(2))
print(f"Remplissage du cache : météo {remplissage_meteo:.1f} s, disponibilite {remplissage_dispo:.1f} s")
print(f"Relevés dans disponibilite : {n_disponibilite:,} ".replace(",", " ")
      + f"({n_propres - n_disponibilite} doublons (station_id, horodatage) écartés)")

Scénario A : [Row(est_pluie=False, taux_moyen=0.3671, releves=10257599), Row(est_pluie=True, taux_moyen=0.3819, releves=511663)]


Scénario B : [Row(est_pluie=False, taux_moyen=0.3671, releves=10257599), Row(est_pluie=True, taux_moyen=0.3819, releves=511663)]


Scénario C : [Row(est_pluie=False, taux_moyen=0.3671, releves=10257599), Row(est_pluie=True, taux_moyen=0.3819, releves=511663)]


,scenario,essai,duree_s,entree_mo,shuffle_mo,stages
0,A,1,9.23,232.21,117.48,16
1,A,2,8.44,232.21,117.48,16
2,A,3,8.34,232.21,117.48,16
3,B,1,8.36,232.15,117.48,16
4,B,2,8.09,232.15,117.48,16
5,B,3,8.06,232.15,117.48,16
6,C,1,0.75,123.51,0.00,14
7,C,2,0.67,123.51,0.00,14
8,C,3,0.54,123.51,0.00,14


Remplissage du cache : météo 0.1 s, disponibilite 9.5 s
Relevés dans disponibilite : 10 769 262 (0 doublons (station_id, horodatage) écartés)


L'onglet *Storage* du Spark UI (ici par l'API REST) montre ce que coûte le cache en mémoire, et le plan de
la requête montre qu'elle lit désormais `InMemoryTableScan` au lieu des fichiers CSV.

In [16]:
import requests

api = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
display(pd.DataFrame([
    {"nom": r["name"][:60], "niveau": r["storageLevel"], "partitions": r["numCachedPartitions"],
     "memoire_mo": r["memoryUsed"] / 1e6, "disque_mo": r["diskUsed"] / 1e6}
    for r in requests.get(f"{api}/storage/rdd", timeout=10).json()
]).round(1))

plan_cache = consolider(disponibilite, meteo, air_horaire)._jdf.queryExecution().executedPlan().toString()
print("\n".join(l.strip()[:110] for l in plan_cache.splitlines()
                if "InMemoryTableScan" in l or "FileScan" in l))

,nom,niveau,partitions,memoire_mo,disque_mo
0,"*(1) Project [time#2068 AS horodatage#2082, te...",Disk Memory Serialized 1x Replicated,1,0.1,0.0
1,AdaptiveSparkPlan isFinalPlan=false\n+- Projec...,Disk Memory Serialized 1x Replicated,8,123.4,0.0


:     :     +- InMemoryTableScan [arrondissement#2048, bornettes_libres#2054, capacite_totale#2055, horodatage
:        +- FileScan csv [horodatage#45,capacite#46,velos_mecaniques#47,velos_electriques#48,nom_station#49,co
+- FileScan parquet [nom_station#1807,latitude#1808,longitude#1809,releves#1810L,arrondissement#1818,station_i
:        +- FileScan csv [horodatage#45,capacite#46,velos_mecaniques#47,velos_electriques#48,nom_station#49,co
+- FileScan parquet [nom_station#1807,latitude#1808,longitude#1809,releves#1810L,arrondissement#1818,station_i
:              +- InMemoryTableScan [est_pluie#2087, horodatage#2082, humidite_pct#2084, precipitation_mm#2085
:                             +- FileScan csv [time#2068,temperature_2m#2069,relative_humidity_2m#2070,precipi
+- FileScan parquet [horodatage#2603,no2_ugm3#2604,pm25_ugm3#2605] Batched: true, DataFilters: [isnotnull(horo


Lecture des mesures :

- **A et B sont identiques** (~8,5 s par exécution) : persister la météo seule ne sert à rien. Elle fait 3 240 lignes (0,1 Mo en cache), se relit en quelques millisecondes et, surtout, elle est déjà **diffusée** (broadcast) à chaque exécution. Le coût est ailleurs : à chaque fois, Spark relit les 232 Mo de gzip, parse le CSV et écrit **117 Mo de shuffle** pour la fenêtre de dédoublonnage (16 stages).
- **C divise le temps par plus de 10** (~0,6 s) : la requête lit `disponibilite`, déjà nettoyée et dédoublonnée, en mémoire (`InMemoryTableScan`). Il n'y a plus ni lecture gzip ni shuffle (14 stages, 0 Mo de shuffle).
- Le cache a un **coût de remplissage** (~10 s, soit une exécution sans cache) : il est rentable dès la deuxième utilisation. C'est le cas ici, puisque la même table sert à toutes les mesures puis à l'écriture Parquet.

Le cache occupe **123 Mo pour 10,8 millions de lignes** (~11 octets par ligne), grâce au stockage en colonnes compressées, contre 460 Mo pour le RDD Python persisté au carnet 01 avec moins de colonnes.

La fenêtre de dédoublonnage n'a écarté **aucune ligne** : une fois les relevés hors service retirés, aucune station n'a deux positions relevées au même instant. On la garde comme garantie d'unicité de `(station_id, horodatage)`, dont dépendent les fonctions `LAG` / `LEAD` du carnet 03. C'est aussi elle qui produit le shuffle, un bon candidat pour l'analyse du carnet 06.

## 11. Écriture en Parquet partitionné par année et par mois

La table consolidée (relevés + calendrier + météo + air) est écrite depuis le cache. `repartition` sur
les colonnes de partition donne un fichier par mois ; le tri par station et horodatage regroupe les
valeurs proches (meilleure compression) et prépare les fonctions de fenêtre des carnets suivants.

In [17]:
sc.setJobDescription("Parquet : écriture de la table consolidée")
shutil.rmtree(PARQUET_CONSOLIDE, ignore_errors=True)
t0 = time.perf_counter()
(consolide
 .repartition("annee", "mois")
 .sortWithinPartitions("station_id", "horodatage")
 .write.partitionBy("annee", "mois")
 .parquet(str(PARQUET_CONSOLIDE)))
t_ecriture = time.perf_counter() - t0

fichiers_parquet = sorted(PARQUET_CONSOLIDE.rglob("*.parquet"))
taille_parquet = sum(f.stat().st_size for f in fichiers_parquet)
taille_gzip = sum(f.stat().st_size for f in VELIB_RAW.glob("velib_*.csv.gz"))
print(f"Écriture en {t_ecriture:.1f} s : {len(fichiers_parquet)} fichiers, {taille_parquet / 1e6:.0f} Mo "
      f"(CSV gzip : {taille_gzip / 1e6:.0f} Mo)")
for f in fichiers_parquet:
    print(f"  {f.parent.parent.name}/{f.parent.name:<8} {f.stat().st_size / 1e6:6.1f} Mo")

Écriture en 9.3 s : 6 fichiers, 149 Mo (CSV gzip : 232 Mo)
  annee=2020/mois=11     6.6 Mo
  annee=2020/mois=12    39.2 Mo
  annee=2021/mois=1     22.1 Mo
  annee=2021/mois=2     30.9 Mo
  annee=2021/mois=3     38.4 Mo
  annee=2021/mois=4     12.0 Mo


On relit la table : le schéma est conservé (Parquet stocke les types), les colonnes de partition
reviennent depuis les noms de dossiers, et un filtre sur `annee` / `mois` n'ouvre que les dossiers
concernés (**partition pruning**, visible dans `PartitionFilters`).

In [18]:
table = spark.read.parquet(str(PARQUET_CONSOLIDE))
table.printSchema()
print(f"Partitions à la lecture : {table.rdd.getNumPartitions()}")

sc.setJobDescription("Parquet : comptage complet")
t0 = time.perf_counter()
n_table = table.count()
t_complet = time.perf_counter() - t0

janvier = table.filter((F.col("annee") == 2021) & (F.col("mois") == 1))
sc.setJobDescription("Parquet : comptage de janvier 2021")
t0 = time.perf_counter()
n_janvier = janvier.count()
t_janvier = time.perf_counter() - t0

scan = [l for l in janvier._jdf.queryExecution().executedPlan().toString().splitlines() if "FileScan" in l][0]
print(f"Table complète : {n_table:,} lignes en {t_complet:.2f} s | janvier 2021 : {n_janvier:,} lignes "
      f"en {t_janvier:.2f} s".replace(",", " "))
print("PartitionFilters" + scan.split("PartitionFilters")[1].split("]")[0] + "]")
assert n_table == n_disponibilite

root
 |-- horodatage_heure: timestamp (nullable = true)
 |-- station_id: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- arrondissement: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- horodatage: timestamp (nullable = true)
 |-- velos_mecaniques: integer (nullable = true)
 |-- velos_electriques: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- capacite_totale: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- date: date (nullable = true)
 |-- heure: integer (nullable = true)
 |-- jour_semaine: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- vitesse_vent_kmh: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- no2_ugm3: double (nullable = true)
 |-- pm25_ugm3

Le Parquet occupe **149 Mo, contre 232 Mo** pour le gzip, alors qu'il contient 25 colonnes au lieu de 7. Le stockage en colonnes, trié par station et horodatage, compresse très bien les valeurs répétées : la météo est la même pour toutes les stations à la même heure, et le nom comme l'arrondissement sont constants pour une station. Le schéma est relu tel quel, sans inférence.

Les comptages sont quasi instantanés, parce que Parquet stocke le nombre de lignes dans ses métadonnées. Le filtre sur janvier 2021 n'ouvre qu'un dossier (`PartitionFilters`). À la lecture, Spark découpe les fichiers par blocs (9 partitions pour 6 fichiers) : contrairement au gzip, Parquet est **découpable**, et le déséquilibre entre mois (6,6 Mo en novembre, 39 Mo en décembre) ne fige plus le nombre de tâches.

## 12. Mesures et bilan

Les mesures rejoignent `mesures.json` pour le bilan du carnet 06.

In [19]:
import json

moyennes = tableau_cache[tableau_cache.essai > 1].groupby("scenario").duree_s.mean()
enregistrer_mesures("02_dataframe_pipeline", {
    "releves_bruts": n_brut,
    "releves_conserves": n_propres,
    "releves_disponibilite": n_disponibilite,
    "definition_inferschema_s": round(t_inference, 2),
    "definition_schema_explicite_s": round(t_explicite, 3),
    "jointure_sans_cache_s": round(tableau_cache[tableau_cache.scenario == "A"].duree_s.mean(), 2),
    "jointure_meteo_cache_s": round(moyennes["B"], 2),
    "jointure_meteo_et_disponibilite_cache_s": round(moyennes["C"], 2),
    "remplissage_cache_disponibilite_s": round(remplissage_dispo, 2),
    "ecriture_parquet_s": round(t_ecriture, 2),
    "parquet_mo": round(taille_parquet / 1e6, 1),
    "csv_gzip_mo": round(taille_gzip / 1e6, 1),
    "comptage_parquet_complet_s": round(t_complet, 2),
    "comptage_parquet_un_mois_s": round(t_janvier, 2),
})
print(json.dumps(json.loads(MESURES_JSON.read_text(encoding="utf-8"))["02_dataframe_pipeline"], indent=2))

{
  "releves_bruts": 10986730,
  "releves_conserves": 10769262,
  "releves_disponibilite": 10769262,
  "definition_inferschema_s": 3.8,
  "definition_schema_explicite_s": 0.163,
  "jointure_sans_cache_s": 8.67,
  "jointure_meteo_cache_s": 8.08,
  "jointure_meteo_et_disponibilite_cache_s": 0.61,
  "remplissage_cache_disponibilite_s": 9.52,
  "ecriture_parquet_s": 9.31,
  "parquet_mo": 149.2,
  "csv_gzip_mo": 231.9,
  "comptage_parquet_complet_s": 0.11,
  "comptage_parquet_un_mois_s": 0.05
}


## Bilan du Jour 1 après-midi

| Étape | Résultat |
|---|---|
| Schéma | schéma explicite : types cibles, sans lecture préalable (`inferSchema` coûte 3,8 s et lit l'horodatage comme une chaîne) |
| Nettoyage | 98,0 % des relevés gardés ; rejets : hors service (2 %), coordonnées fausses (680) ; 2 005 taux bornés à 1 |
| Référentiel | 1 384 positions -> 1 382 stations : 1 301 par le nom, 36 par la position, 47 identifiants de substitution |
| `disponibilite` | 10 769 262 relevés au schéma cible, `(station_id, horodatage)` unique |
| `meteo_horaire` | 3 240 heures complètes, 5,7 % d'heures de pluie |
| Qualité de l'air | NO2 et PM2.5 pour les 3 240 heures (31 et 7 points de mesure en moyenne), ramenés de UTC+1 à UTC |
| Jointures | 3 jointures broadcast ; un seul shuffle (dédoublonnage) |
| Cache (contrainte n°6, jointure) | ~8,5 s sans cache, autant avec la météo seule, ~0,6 s avec `disponibilite` persistée |
| Parquet | 149 Mo en 6 partitions mensuelles ; comptage d'un mois par élagage des partitions |

La seconde mesure de la contrainte n°6, sur un **calcul itératif**, sera faite avec le K-Means du carnet 05. Fichiers produits pour la suite : `disponibilite_parquet` (table consolidée), `stations`, `meteo_horaire` et `air_horaire`.

On libère le cache puis la session.

In [20]:
disponibilite.unpersist()
meteo.unpersist()
spark.stop()
print("SparkSession arrêtée.")

SparkSession arrêtée.
